# Module 2: Deploy + Context Hub

> Part of the **Modular Workshops** series. Standalone, ~25 min.

Two steps to take the agent to production:

- **Context Hub** — push the agent's context (`AGENTS.md` + skills) to LangSmith as **versioned, shareable repos**, so it's centralized and reusable instead of baked into one codebase.
- **Deployment** — ship the agent to **LangSmith Deployments** with the `langgraph` CLI: 30+ endpoints, persistence, HITL, and Studio out of the box.

Context first, then deploy.

## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(dotenv_path=project_root / ".env", override=True)

import os

print("LANGSMITH_API_KEY set:        ", bool(os.environ.get("LANGSMITH_API_KEY")))
print("OPENAI_API_KEY set:           ", bool(os.environ.get("OPENAI_API_KEY")))
print("TAVILY_API_KEY set:           ", bool(os.environ.get("TAVILY_API_KEY")))

---
# Part 1. Context Hub — Centralize the Agent's Context

An agent's *context* — its `AGENTS.md` instructions and its skills — matters as much as its code. **LangSmith Context Hub** stores that context as **versioned, shareable repos**: push it once, and any deployment (or teammate, or *other* agent) pulls the same source of truth.

We push it in two pieces — each **skill** as its own reusable repo, then the **agent repo** that links them — and read it back. At deploy time the agent loads this context via deepagents' `ContextHubBackend` (mounted at `/context/`).

### 1.1 Push each skill as a reusable repo

`push_skill` creates (or versions) a standalone skill repo. Because it's independent, **any** agent can link it — no copy-paste.

In [ ]:
from pathlib import Path
from langsmith import Client
from langsmith.schemas import FileEntry, SkillEntry

client = Client()
agent_dir = project_root / "agents" / "deep_agent"

skill_urls = {}
for skill_dir in sorted((agent_dir / "skills").iterdir()):
    skill_md = skill_dir / "SKILL.md"
    if not skill_md.exists():
        continue
    skill_urls[skill_dir.name] = client.push_skill(
        skill_dir.name,
        files={"SKILL.md": FileEntry(content=skill_md.read_text())},
        description=f"Reusable '{skill_dir.name}' skill.",
        tags=["workshop"],
    )

for name, url in skill_urls.items():
    print(f"skill '{name}': {url}")

### 1.2 Push the agent context (linking the skills)

`push_agent` uploads `AGENTS.md` and **links** the skill repos with `SkillEntry` instead of inlining them — so a skill update propagates to every agent that links it.

In [ ]:
agent_url = client.push_agent(
    "modular-workshops-research-agent",
    files={
        "AGENTS.md": FileEntry(content=(agent_dir / "AGENTS.md").read_text()),
        **{f"skills/{name}": SkillEntry(repo_handle=name) for name in skill_urls},
    },
    description="Research assistant used in the Modular Workshops deploy module.",
    tags=["workshop", "research"],
)
print("agent context:", agent_url)

### 1.3 Pull it back — and how the deployed agent uses it

`pull_agent` fetches the latest commit (pin a `version=` or tag for reproducibility). At deploy time, `agents/deep_agent/agent.py` mounts a Context Hub repo at `/context/` through deepagents' `ContextHubBackend`, so the running agent reads this same context — centralized and versioned, not baked into the image.

In [ ]:
pulled = client.pull_agent("modular-workshops-research-agent")
print("commit:", pulled.commit_hash)
print("files:", list(pulled.files))

---
# Part 2. Deploy — Ship the Agent

With the context in the Hub, ship the deployment itself. We deploy the agent in `agents/deep_agent/` (and your own `my_agent/`) to **LangSmith Deployments** with the `langgraph` CLI. The deployed agent can pull its context from the Hub at runtime via `ContextHubBackend`.

## 2.1 Project structure

A deployable LangGraph project is a directory with a `langgraph.json` config at the root that points at one or more graph objects. We already have one — `langgraph.json` at the workshop root registers the deep agent at `agents/deep_agent/agent.py`. You'll add a second graph — **your own agent** — in §2.5, and a single deploy ships both.


In [ ]:
import os

agent_dir = str(project_root / "agents" / "deep_agent")
print("langgraph.json (workshop root)")
print("---")

for root, dirs, files in os.walk(agent_dir):
    # Skip __pycache__ for clarity
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(agent_dir, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")


In [ ]:
# langgraph.json -- the deploy configuration
langgraph_json_path = project_root / "langgraph.json"
with open(langgraph_json_path) as f:
    print("langgraph.json:")
    print(f.read())

# AGENTS.md -- the agent's identity
agents_path = os.path.join(agent_dir, "AGENTS.md")
with open(agents_path) as f:
    print("AGENTS.md:")
    print(f.read())


The agent itself lives in `agent.py`. The module-level `agent` variable is what gets deployed — `langgraph.json` references it as `"deep_agent": "./agents/deep_agent/agent.py:agent"`.


## 2.3 Local development

Three CLI commands you'll use (all run from the workshop root, where `langgraph.json` lives):

```bash
# Validate langgraph.json (imports each graph, checks deps)
langgraph validate

# Run locally for development (Studio UI + hot reload)
langgraph dev --port 2024

# Deploy to LangSmith (beta)
langgraph deploy
```

`langgraph dev` opens the LangGraph Studio UI in your browser — useful to step through tool calls and approve HITL interrupts visually. By default it connects to `https://smith.langchain.com` for the Studio frontend and talks to your local server.

**Docs:** [LangGraph CLI reference](https://docs.langchain.com/langsmith/langgraph-cli) · [Deploy on Cloud](https://docs.langchain.com/langsmith/deploy-to-cloud).


## 2.4 Chat with your agent (Agent Chat UI)

`langgraph dev` doesn't just power Studio — it also backs a ready-made **chat frontend**, with zero code on your side. [Agent Chat UI](https://github.com/langchain-ai/agent-chat-ui) is a hosted chat app that talks to any LangGraph server exposing a `messages` key, which our `deep_agent` does. Point it at the local dev server and you have a real UI to demo the agent in.

**1. Start the dev server** — in a separate terminal, from the workshop root, and leave it running:

```bash
langgraph dev --port 2024
```

**2. Open the hosted UI:** <https://agentchat.vercel.app>

**3. Fill in the connection form:**

| Field | Value |
|---|---|
| Deployment URL | `http://localhost:2024` |
| Assistant / Graph ID | `deep_agent` |
| LangSmith API Key | *(leave blank — not needed for a local server)* |

**4. Chat.** Messages stream token-by-token and tool calls render inline. Because the agent gates its file tools with human-in-the-loop (`interrupt_on={"write_file", "edit_file"}`), Agent Chat UI surfaces an **approval prompt** — accept, edit, or respond — before the agent writes or edits a file.

> **Nothing leaves your laptop.** The hosted app runs entirely in your browser and connects directly to `localhost:2024` on your machine — browsers treat `http://localhost` as a secure origin, so an `https://` page can reach it with no proxy for local dev.
>
> **Prefer a fully local UI?** `npx create-agent-chat-app@latest` scaffolds the same app (Next.js or Vite) on `http://localhost:3000` with the connection pre-filled. Going to **production** is different — you don't point the browser straight at the server; you host the UI and proxy requests (or add custom auth). See the [Agent Chat UI docs](https://docs.langchain.com/oss/python/langgraph/ui).


## 2.5 Create your own deep agent

Your turn. `my_agent/agent.py` is a blank deep-agent scaffold — a `create_deep_agent` with one search tool and a research subagent, plus commented stubs for extra tools, subagents, backends, middleware, skills, and human-in-the-loop. Make it yours:

1. Open **`my_agent/agent.py`** and edit the system prompt, add tools, wire up subagents — whatever your use case needs. (`my_agent/AGENTS.md` is its identity file.)
2. It's **already registered** in `langgraph.json` as the `my_agent` graph, so you can visualize it with the tools from the last two sections:
   - `langgraph dev --port 2024` (§2.3) picks it up — open **Studio** and pick `my_agent` to step through it, or point **Agent Chat UI** (§2.4) at graph id `my_agent` to chat with it.
   - the deploy step (§2.7) ships it **alongside** `deep_agent` in one deployment.

Run the cell to smoke-test that your agent builds and responds:


In [ ]:
# Reload so edits to my_agent/agent.py are picked up without restarting the kernel.
import importlib, my_agent.agent
importlib.reload(my_agent.agent)
from my_agent.agent import agent as my_agent_graph

result = my_agent_graph.invoke(
    {"messages": [{"role": "user", "content": "In one sentence, introduce yourself and what you can do."}]}
)
print(result["messages"][-1].content)


## 2.6 Validate the config

`langgraph validate` imports each graph in `langgraph.json` and verifies dependencies resolve — without building Docker or uploading anything. Use it to catch config and import errors before deploying.


In [ ]:
# `cd` and `!` run in a subshell — chain in one line so cwd applies to the langgraph command.
!cd "{project_root}" && langgraph validate


## 2.7 Deploy to LangSmith (optional)

Run the cell below to deploy to **LangSmith Deployments**. `langgraph deploy` ships **every graph** in `langgraph.json`, so `deep_agent` and your `my_agent` go out together in one deployment. `langgraph deploy` builds a Docker image (locally if Docker is available, otherwise remotely on LangSmith's builder) and pushes it. Provisioning takes a few minutes.


> Requires a `LANGSMITH_API_KEY` with deployment permissions — a service key (`lsv2_sk_...`), not a personal token. On Apple Silicon, local builds need Docker Buildx; without Docker, the CLI falls back to a remote build automatically.

Useful flags:
- `--name <name>` — deployment name (defaults to the project directory name)
- `--deployment-type prod` — production deployment (default is `dev`)
- `--remote` — force remote build, skip local Docker
- `--no-wait` — return immediately rather than blocking on status

In [ ]:
# Re-run this command to push a new revision; the CLI finds the existing deployment by name.
# Add `--deployment-type prod` for production, or `--remote` to skip local Docker.
!cd "{project_root}" && langgraph deploy --name modular-workshops-deep-agent --no-input


## 2.8 What you get with LangSmith Deployments

Once deployed, your agent is reachable through 30+ endpoints — you build it once, the platform exposes it everywhere:

| Capability | What you can do |
|---|---|
| **REST API** | Standard HTTP requests against `/runs`, `/threads`, `/store` |
| **Studio UI** | Visual debugger to step through state, threads, and tool calls |
| **Agent Protocol** | Stream runs and pause for human input |
| **MCP server** | Other agents can call your agent as a tool |
| **A2A** | Agent-to-agent calls with handoffs |
| **Persistent Store** | `/memories/` survives restarts and threads (via the platform's Store) |
| **HITL** | Interrupt and resume from any client |
| **Cron / Scheduled runs** | Trigger your agent on a schedule |
